# NB4 — Embeddings denses et embeddings de phrases avec tuning intégré

Ce notebook couvre `P16` à `P20`. La phase de tuning accéléré est particulièrement utile ici, car les embeddings denses peuvent coûter cher à recalculer ; on les calcule donc une seule fois par pipeline dans la phase d’optimisation.

Ce notebook conserve la **phase baseline sans optimisation**, puis ajoute une **phase d’optimisation accélérée**. 
Le principe retenu est le suivant : pour chaque pipeline, on ajuste d’abord le **préprocesseur / vectoriseur une seule fois** sur un sous-ensemble d’apprentissage, puis on teste plusieurs réglages du **classifieur uniquement** sur les mêmes données déjà transformées. Cela réduit fortement le temps d’exécution.

Cette stratégie est très pratique pour explorer rapidement des réglages d’algorithmes, mais il faut bien comprendre qu’elle constitue une **optimisation accélérée**, plus pragmatique qu’une recherche entièrement relancée sur tout le pipeline à chaque itération.


In [ ]:
# Pour un run sur Colab

'''
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = "/content/drive/MyDrive/Disaster-Tweets-NLP"
MODELS_DIR = f"{PROJECT_ROOT}/notebooks/models_training"

%cd "{MODELS_DIR}"

import sys
if MODELS_DIR not in sys.path:
    sys.path.append(MODELS_DIR)

print("Projet :", PROJECT_ROOT)
print("Dossier courant :", MODELS_DIR)
'''


In [ ]:
# Installation éventuelle (décommente si nécessaire)
# !pip install pandas numpy scikit-learn scipy matplotlib gensim sentence-transformers openpyxl mlflow

import json
from collections import OrderedDict
from pathlib import Path

import pandas as pd
from mlflow_utils import (
    fit_evaluate_and_log_sklearn_pipeline,
    setup_mlflow_tracking,
)
from nlp_disaster_utils import (
    GensimMeanEmbeddingVectorizer,
    SentenceTransformerVectorizer,
    load_train_test_xy,
    round_results,
    save_results_bundle,
    seed_everything,
    stratified_validation_split,
)
from pipeline_tuning_utils import (
    compare_baseline_vs_tuned,
    evaluate_refit_outputs,
    fit_transform_preprocessor_once,
    log_tuning_run_to_mlflow,
    safe_scores,
    split_pipeline_preprocessor_estimator,
    tune_classifier_on_fixed_features,
)
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC


seed_everything(42)


In [ ]:
DATA_DIR = "../../data/processed_data"
TRAIN_PATH = f"{DATA_DIR}/train.csv"
TEST_PATH = f"{DATA_DIR}/test.csv"

TEXT_COL = "text"
LABEL_COL = "target"
USE_AUX_TEXT_COLUMNS = False
LOWERCASE_TEXT = False
RANDOM_STATE = 42

OUTPUT_STEM = "NB4_classical_sentence_embeddings"
RESULTS_DIR = "../../outputs/NB4"
TUNING_OUTPUT_DIR = Path(RESULTS_DIR) / "tuning"
TUNING_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Configuration MLflow
MLFLOW_EXPERIMENT_NAME = "DT_NB4_classical_sentence_embeddings"
MLFLOW_TRACKING_URI = Path("../../outputs/mlruns").resolve().as_uri()
MLFLOW_LOG_MODEL = False
USE_MLFLOW = True

tracking_uri = setup_mlflow_tracking(
    experiment_name=MLFLOW_EXPERIMENT_NAME,
    tracking_uri=MLFLOW_TRACKING_URI,
)
print("MLflow tracking URI :", tracking_uri)
print("MLflow experiment   :", MLFLOW_EXPERIMENT_NAME)

# Paramètres de tuning accéléré
VAL_SIZE_FOR_TUNING = 0.15
PRIMARY_TUNING_METRIC = "f1_pos"


In [ ]:
df_train, X_train, y_train, df_test, X_test, y_test = load_train_test_xy(
    train_path=TRAIN_PATH,
    test_path=TEST_PATH,
    text_col=TEXT_COL,
    label_col=LABEL_COL,
    use_extra_cols=USE_AUX_TEXT_COLUMNS,
    lowercase=LOWERCASE_TEXT,
)

print("Taille train :", len(X_train))
print("Taille test  :", len(X_test))
print("\nDistribution des classes - train :")
print(y_train.value_counts(normalize=True).sort_index())
print("\nDistribution des classes - test :")
print(y_test.value_counts(normalize=True).sort_index())


In [ ]:
pipelines = OrderedDict({
    "P16_GloVeTwitterMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P17_GloVeTwitterMean_LinearSVC": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="glove-twitter-200", normalize=True)),
        ("clf", LinearSVC(C=1.0)),
    ]),
    "P18_FastTextMean_LogReg": Pipeline([
        ("embed", GensimMeanEmbeddingVectorizer(model_name="fasttext-wiki-news-subwords-300", normalize=True)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P19_SentenceTransformer_LogReg": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LogisticRegression(max_iter=2500, C=1.0)),
    ]),
    "P20_SentenceTransformer_LinearSVC": Pipeline([
        ("embed", SentenceTransformerVectorizer(model_name="sentence-transformers/all-MiniLM-L6-v2", batch_size=64)),
        ("clf", LinearSVC(C=1.0)),
    ]),
})


## Phase 1 — Baselines sans optimisation

Cette première phase reproduit le benchmark initial : chaque pipeline est exécuté tel quel, avec ses paramètres de départ.

In [ ]:
resultats = []
baseline_failures = []

for nom_pipeline, pipeline in pipelines.items():
    print(f"Entraînement baseline -> {nom_pipeline}")
    display(pipeline)
    print("-" * 80)

    try:
        metrics = fit_evaluate_and_log_sklearn_pipeline(
            name=nom_pipeline,
            estimator=pipeline,
            X_train=X_train,
            X_test=X_test,
            y_train=y_train,
            y_test=y_test,
            notebook_name="CLASSICAL",
            family_name="classical_sentence_embeddings",
            output_dir=RESULTS_DIR,
            log_model=MLFLOW_LOG_MODEL,
        )
        resultats.append(metrics)
    except Exception as exc:
        baseline_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec baseline pour {nom_pipeline} : {exc}")

baseline_df = round_results(pd.DataFrame(resultats))
display(baseline_df)

if baseline_failures:
    print("\nPipelines baseline en échec :")
    display(pd.DataFrame(baseline_failures))


In [ ]:
save_results_bundle(pd.DataFrame(resultats), output_dir=RESULTS_DIR, stem=OUTPUT_STEM)
print(f"Fichiers CSV/XLSX baseline enregistrés dans {RESULTS_DIR}")


## Phase 2 — Tuning accéléré avec vectorisation unique par pipeline

Ici, pour chaque pipeline, on sépare le **préprocesseur** du **classifieur**. On ajuste le préprocesseur **une seule fois** sur un sous-ensemble d’apprentissage, puis on teste différentes combinaisons d’hyperparamètres du classifieur sur les mêmes données déjà vectorisées. Enfin, on réajuste le meilleur classifieur sur tout le train transformé une seule fois et on l’évalue sur train et test.

In [ ]:
X_fit, X_val, y_fit, y_val = stratified_validation_split(
    X_train,
    y_train,
    val_size=VAL_SIZE_FOR_TUNING,
    random_state=RANDOM_STATE,
)

print("Taille tuning-fit :", len(X_fit))
print("Taille tuning-val :", len(X_val))


In [ ]:
classifier_param_grids = {
    "P16_GloVeTwitterMean_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P17_GloVeTwitterMean_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P18_FastTextMean_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P19_SentenceTransformer_LogReg": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
    "P20_SentenceTransformer_LinearSVC": {
        "C": [0.25, 0.5, 1.0, 2.0],
        "class_weight": [None, "balanced"],
    },
}


In [ ]:
tuning_rows = []
tuning_failures = []
tuned_metrics_rows = []

for nom_pipeline, pipeline in pipelines.items():
    print("=" * 100)
    print(f"Tuning accéléré -> {nom_pipeline}")

    param_grid = classifier_param_grids.get(nom_pipeline)
    if param_grid is None:
        tuning_failures.append({"pipeline": nom_pipeline, "error": "Grille d'hyperparamètres absente"})
        print("Aucune grille trouvée.")
        continue

    try:
        preprocessor, clf_name, base_estimator = split_pipeline_preprocessor_estimator(pipeline)

        transformed = fit_transform_preprocessor_once(
            preprocessor=preprocessor,
            X_fit=X_fit,
            y_fit=y_fit,
            X_val=X_val,
        )

        best_params, tuning_results_df = tune_classifier_on_fixed_features(
            base_estimator=base_estimator,
            param_grid=param_grid,
            X_fit=transformed["X_fit_transformed"],
            y_fit=y_fit,
            X_val=transformed["X_val_transformed"],
            y_val=y_val,
            primary_metric=PRIMARY_TUNING_METRIC,
        )

        tuning_results_df.insert(0, "pipeline", nom_pipeline)
        tuning_results_df.insert(1, "classifier_name", clf_name)

        best_preprocessor_full, _, best_estimator_template = split_pipeline_preprocessor_estimator(pipeline)
        best_preprocessor_full.fit(X_train, y_train)
        X_train_vec = best_preprocessor_full.transform(X_train)
        X_test_vec = best_preprocessor_full.transform(X_test)

        best_estimator = base_estimator.set_params(**best_params)
        best_estimator.fit(X_train_vec, y_train)

        train_pred = best_estimator.predict(X_train_vec)
        test_pred = best_estimator.predict(X_test_vec)
        train_score = safe_scores(best_estimator, X_train_vec)
        test_score = safe_scores(best_estimator, X_test_vec)

        final_metrics = evaluate_refit_outputs(
            pipeline_name=nom_pipeline,
            y_train=y_train,
            y_test=y_test,
            train_pred=train_pred,
            test_pred=test_pred,
            train_score=train_score,
            test_score=test_score,
        )
        final_metrics["best_params"] = json.dumps(best_params, ensure_ascii=False)
        final_metrics["best_val_primary_score"] = float(tuning_results_df.iloc[0]["primary_score"])
        final_metrics["best_val_f1_class_1"] = float(tuning_results_df.iloc[0]["val_f1_class_1"])
        final_metrics["best_val_recall_class_1"] = float(tuning_results_df.iloc[0]["val_recall_class_1"])
        final_metrics["best_val_f1_macro"] = float(tuning_results_df.iloc[0]["val_f1_macro"])
        final_metrics["best_val_balanced_accuracy"] = float(tuning_results_df.iloc[0]["val_balanced_accuracy"])

        tuning_rows.append(tuning_results_df.iloc[0].to_dict() | {
            "pipeline": nom_pipeline,
            "best_params": json.dumps(best_params, ensure_ascii=False),
        })
        tuned_metrics_rows.append(final_metrics)

        tuning_results_path = TUNING_OUTPUT_DIR / f"{nom_pipeline}_tuning_validation_results.csv"
        tuning_results_df.to_csv(tuning_results_path, index=False)

        if USE_MLFLOW:
            log_tuning_run_to_mlflow(
                run_name=nom_pipeline,
                notebook_name="CLASSICAL",
                family_name="classical_sentence_embeddings",
                best_params=best_params,
                tuning_results_df=tuning_results_df,
                final_metrics=final_metrics,
                output_dir=TUNING_OUTPUT_DIR,
            )

        print("Meilleurs paramètres :", best_params)
        print("Meilleur score de validation :", tuning_results_df.iloc[0]["primary_score"])

    except Exception as exc:
        tuning_failures.append({"pipeline": nom_pipeline, "error": str(exc)})
        print(f"Échec tuning pour {nom_pipeline} : {exc}")


In [ ]:
tuning_best_df = round_results(pd.DataFrame(tuning_rows))
display(tuning_best_df)

tuned_results_df = round_results(pd.DataFrame(tuned_metrics_rows))
display(tuned_results_df)

if tuning_failures:
    print("\nPipelines tuning en échec :")
    display(pd.DataFrame(tuning_failures))


In [ ]:
comparison_df = compare_baseline_vs_tuned(
    baseline_df=pd.DataFrame(resultats),
    tuned_df=pd.DataFrame(tuned_metrics_rows),
)
display(round_results(comparison_df))


In [ ]:
pd.DataFrame(tuning_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_resume.csv", index=False)
pd.DataFrame(tuned_metrics_rows).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuned_final_results.csv", index=False)
pd.DataFrame(tuning_failures).to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_tuning_failures.csv", index=False)

comparison_df.to_csv(TUNING_OUTPUT_DIR / f"{OUTPUT_STEM}_baseline_vs_tuned.csv", index=False)

print("Exports tuning enregistrés dans :", TUNING_OUTPUT_DIR)


Si certaines dépendances comme PyTorch ou SentenceTransformer posent problème localement, les pipelines concernés peuvent échouer dans la phase baseline ou tuning, mais les autres continueront de s’exécuter et d’être exportés.